# RELAX Global SHAP — Numerical Reproduction

**Purpose.** Re-execute only the RELAX global SHAP analysis from the original `PHASE_III_Model_Aligned_SHAP_Stability_FIXED.ipynb`, preserve the archived reference outputs, and compare the new results fold by fold.

**This notebook does not change the analysis design.** It retains the original five-fold `GroupKFold`, fold-specific seeds, participant-equal training weights, median imputation with missingness indicators, 400-tree Random Forest, `TreeExplainer`, mean absolute SHAP aggregation, and Spearman/Kendall/Top-5 Jaccard calculations.

**Important:** the original run recorded Python 3.13.18, pandas 3.1.1, scikit-learn 1.9.1 and SHAP 0.54.1. NumPy and SciPy were not recorded. This notebook records the environment actually used. If it differs from the recorded versions, label the run diagnostic rather than exact historical reproduction. Do not overwrite archived references or manuscript results.

## How to run
1. Open this notebook in Google Colab (or Jupyter with the required packages).
2. Run cells from top to bottom.
3. When prompted, upload `RELAX_Global_SHAP_Notebook_Inputs.zip`.
4. Allow all five folds to finish.
5. Inspect the comparison and integrity-check cells.
6. Download the generated results ZIP at the end.

## 1. Imports and environment record

Do not upgrade the full scientific Python stack silently. The imports below use the environment already active in the notebook. If an import fails, install the missing package explicitly, restart the runtime, and record the resulting versions.

In [ ]:
from pathlib import Path
import os, sys, json, zipfile, hashlib, platform, warnings, itertools, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import sklearn
from scipy.stats import spearmanr, kendalltau
from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

ORIGINAL_ENV = {
    "python": "3.13.18", "pandas": "3.1.1",
    "scikit_learn": "1.9.1", "shap": "0.54.1",
    "numpy": None, "scipy": None
}
CURRENT_ENV = {
    "python": platform.python_version(), "python_full": sys.version,
    "platform": platform.platform(), "numpy": np.__version__,
    "pandas": pd.__version__, "scipy": __import__("scipy").__version__,
    "scikit_learn": sklearn.__version__, "shap": shap.__version__
}
print(json.dumps(CURRENT_ENV, indent=2))
print("Recorded historical versions:", json.dumps(ORIGINAL_ENV, indent=2))

## 2. Upload and extract the frozen inputs

Upload the companion archive `RELAX_Global_SHAP_Notebook_Inputs.zip`. It contains the exact source CSV and immutable reference outputs.

In [ ]:
# Google Colab upload; in local Jupyter, set INPUT_ZIP to the path of the same ZIP.
try:
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zip_names:
        raise FileNotFoundError("Upload RELAX_Global_SHAP_Notebook_Inputs.zip")
    INPUT_ZIP = Path(zip_names[0])
except ImportError:
    INPUT_ZIP = Path("RELAX_Global_SHAP_Notebook_Inputs.zip")
    if not INPUT_ZIP.exists():
        raise FileNotFoundError("Place RELAX_Global_SHAP_Notebook_Inputs.zip beside this notebook.")

WORK_DIR = Path.cwd() / "RELAX_Global_SHAP_Controlled_Notebook_Run"
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)
with zipfile.ZipFile(INPUT_ZIP) as zf:
    bad_member = zf.testzip()
    if bad_member is not None:
        raise RuntimeError(f"Corrupt ZIP member: {bad_member}")
    zf.extractall(WORK_DIR / "inputs")

DATA_PATH = WORK_DIR / "inputs" / "data" / "RELAX_next_morning_stress_dataset.csv"
REFERENCE_DIR = WORK_DIR / "inputs" / "reference"
OUTPUT_DIR = WORK_DIR / "reproduced_outputs"
OUTPUT_DIR.mkdir(parents=True)
assert DATA_PATH.exists(), DATA_PATH
assert (REFERENCE_DIR / "RELAX_global_SHAP_summary.csv").exists()
print("Input archive:", INPUT_ZIP)
print("Dataset:", DATA_PATH)
print("Reference directory:", REFERENCE_DIR)
print("New output directory:", OUTPUT_DIR)

## 3. Freeze and validate the dataset

The expected SHA-256 comes from the supplied Phase I archive and Phase III run metadata. The notebook stops if the input differs.

In [ ]:
EXPECTED_SHA256 = "864b0c9b52f123088f46d932b2daf5bb7410c7932fb5e0e233d6e5c51c5ee5ac"
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

input_sha256 = sha256_file(DATA_PATH)
print("Dataset SHA-256:", input_sha256)
if input_sha256 != EXPECTED_SHA256:
    raise ValueError("Dataset hash mismatch. Stop: do not run on a different input.")

relax = pd.read_csv(DATA_PATH)
required = {"participant_id", "target_timestamp", "target_stress_expectation_1_7"}
missing = required - set(relax.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")
relax = relax.copy()
relax["participant_id"] = relax["participant_id"].astype(str)
relax["target_timestamp"] = pd.to_datetime(relax["target_timestamp"], errors="coerce")
relax = relax.dropna(subset=["participant_id", "target_stress_expectation_1_7"]).reset_index(drop=True)
relax_target = "target_stress_expectation_1_7"
relax_exclude = {"participant_id", "target_timestamp", relax_target}
relax_features = [c for c in relax.columns if c not in relax_exclude]
X_relax = relax[relax_features].apply(pd.to_numeric, errors="coerce")
y_relax = pd.to_numeric(relax[relax_target], errors="coerce").to_numpy(float)
g_relax = relax["participant_id"].to_numpy(str)

print("Rows:", len(relax), "| columns:", relax.shape[1], "| participants:", len(np.unique(g_relax)), "| predictors:", len(relax_features))
assert relax.shape == (1148, 37), f"Unexpected shape: {relax.shape}"
assert len(np.unique(g_relax)) == 31
assert len(relax_features) == 34
assert np.isfinite(y_relax).all()
assert len(np.unique(g_relax)) >= 5

## 4. Original method configuration and helper functions

These settings mirror the RELAX global section of the original Phase III notebook. `SEED_RELAX_GLOBAL` is the base seed; the model seed is `20261013 + fold`, giving 20261014–20261018.

In [ ]:
SEED_RELAX_GLOBAL = 20261013
N_SPLITS = 5
TOP_K = 5
RELAX_RF = dict(n_estimators=400, min_samples_leaf=5, max_features=0.8, n_jobs=-1)

def participant_equal_weights(groups):
    groups = np.asarray(groups).astype(str)
    counts = pd.Series(groups).value_counts()
    w = np.array([1.0 / counts[g] for g in groups], dtype=float)
    return w / w.mean()

def shap_values_2d(model, X_transformed):
    vals = shap.TreeExplainer(model).shap_values(X_transformed, check_additivity=False)
    vals = np.asarray(vals)
    if vals.ndim == 3:
        vals = vals[..., 0]
    if vals.ndim != 2:
        raise ValueError(f"Unexpected SHAP array shape: {vals.shape}")
    return vals

def add_pairwise_rank_metrics(importance_wide, dataset_name, top_k=5):
    rank_wide = importance_wide.rank(axis=0, ascending=False, method="average")
    records = []
    fold_ids = list(importance_wide.columns)
    for fa, fb in itertools.combinations(fold_ids, 2):
        pair = pd.DataFrame({"a": rank_wide[fa], "b": rank_wide[fb]}).dropna()
        rho = spearmanr(pair["a"], pair["b"]).statistic if len(pair) > 1 else np.nan
        tau = kendalltau(pair["a"], pair["b"]).statistic if len(pair) > 1 else np.nan
        top_a = set(importance_wide[fa].nlargest(min(top_k, len(importance_wide))).index)
        top_b = set(importance_wide[fb].nlargest(min(top_k, len(importance_wide))).index)
        union = top_a | top_b
        jaccard = len(top_a & top_b) / len(union) if union else np.nan
        records.append({"dataset": dataset_name, "fold_a": fa, "fold_b": fb,
                        "n_features_compared": len(pair), "spearman_rank_correlation": rho,
                        "kendall_rank_correlation": tau, "top_5_jaccard": jaccard})
    return pd.DataFrame(records)

def save_global_outputs(dataset_name, fold_importance_rows):
    fi = pd.DataFrame(fold_importance_rows)
    fi.to_csv(OUTPUT_DIR / f"{dataset_name}_global_SHAP_importance_by_fold.csv", index=False)
    wide = fi.pivot(index="feature", columns="fold", values="mean_abs_shap")
    pairs = add_pairwise_rank_metrics(wide, dataset_name, TOP_K)
    pairs.to_csv(OUTPUT_DIR / f"{dataset_name}_global_SHAP_pairwise_stability.csv", index=False)
    summary = (fi.groupby("feature")
        .agg(mean_abs_shap=("mean_abs_shap", "mean"), sd_abs_shap=("mean_abs_shap", "std"),
             mean_rank=("rank", "mean"), sd_rank=("rank", "std"),
             folds_top5=("rank", lambda x: int((x <= 5).sum())),
             folds_present=("fold", "nunique"))
        .reset_index().sort_values("mean_abs_shap", ascending=False))
    summary.to_csv(OUTPUT_DIR / f"{dataset_name}_global_SHAP_summary.csv", index=False)
    return fi, pairs, summary

## 5. Execute the five-fold RELAX global SHAP analysis

This is the original Phase III RELAX global procedure. It explains every held-out observation in each fold and writes row-level SHAP values for auditability. The reference files are not modified.

In [ ]:
relax_global_importance = []
relax_global_shap_rows = []
relax_global_fold_log = []
relax_splits = list(GroupKFold(n_splits=N_SPLITS).split(X_relax, y_relax, g_relax))

for fold, (tr, te) in enumerate(relax_splits, start=1):
    fold_seed = SEED_RELAX_GLOBAL + fold
    print(f"RELAX global fold {fold}/{N_SPLITS}: seed={fold_seed}; train={len(tr)}; held-out={len(te)}")
    imputer = SimpleImputer(strategy="median", add_indicator=True)
    Xtr = imputer.fit_transform(X_relax.iloc[tr])
    Xte = imputer.transform(X_relax.iloc[te])
    feat_names = list(relax_features)
    if imputer.indicator_ is not None:
        feat_names += [f"missing__{relax_features[i]}" for i in imputer.indicator_.features_]

    model = RandomForestRegressor(**RELAX_RF, random_state=fold_seed)
    model.fit(Xtr, y_relax[tr], sample_weight=participant_equal_weights(g_relax[tr]))
    sv = shap_values_2d(model, Xte)
    if sv.shape != (len(te), len(feat_names)):
        raise ValueError(f"Fold {fold}: SHAP shape {sv.shape}; expected {(len(te), len(feat_names))}")
    if not np.isfinite(sv).all():
        raise ValueError(f"Fold {fold}: non-finite SHAP values")
    mean_abs = np.abs(sv).mean(axis=0)
    ranks = pd.Series(mean_abs, index=feat_names).rank(ascending=False, method="average")
    for f, val, rank in zip(feat_names, mean_abs, ranks):
        relax_global_importance.append({"fold": fold, "feature": f, "mean_abs_shap": float(val),
            "rank": float(rank), "n_test_rows_explained": int(len(te)),
            "n_test_participants": int(len(np.unique(g_relax[te])))})
    # Retain original row-level output structure and one-row prediction calls.
    for j, row_idx in enumerate(te):
        rec = {"fold": fold, "row_index": int(row_idx), "participant_id": g_relax[row_idx],
               "target_stress_expectation_1_7": float(y_relax[row_idx]),
               "predicted_stress_expectation_1_7": float(model.predict(Xte[j:j+1])[0])}
        rec.update({f"shap__{fn}": float(v) for fn, v in zip(feat_names, sv[j])})
        relax_global_shap_rows.append(rec)
    relax_global_fold_log.append({"dataset": "RELAX", "fold": fold,
        "train_rows": len(tr), "test_rows": len(te),
        "train_participants": len(np.unique(g_relax[tr])),
        "test_participants": len(np.unique(g_relax[te])),
        "n_estimators": RELAX_RF["n_estimators"], "seed": fold_seed})

relax_fi, relax_pairs, relax_summary = save_global_outputs("RELAX", relax_global_importance)
pd.DataFrame(relax_global_shap_rows).to_csv(OUTPUT_DIR / "RELAX_global_SHAP_heldout_values.csv", index=False)
relax_fold_audit = pd.DataFrame(relax_global_fold_log)
relax_fold_audit.to_csv(OUTPUT_DIR / "RELAX_global_SHAP_fold_audit.csv", index=False)
print("FOLD AUDIT"); display(relax_fold_audit)
print("PAIRWISE STABILITY"); display(relax_pairs)
print("TOP 15 FEATURES"); display(relax_summary.head(15))
print("Held-out SHAP rows:", len(relax_global_shap_rows))

## 6. Visualise the newly reproduced global results

In [ ]:
top = relax_summary.head(15).sort_values("mean_abs_shap")
plt.figure(figsize=(10, 7))
plt.barh(top["feature"], top["mean_abs_shap"], xerr=top["sd_abs_shap"])
plt.xlabel("Mean absolute SHAP across participant-held-out folds")
plt.ylabel("Feature")
plt.title("RELAX: global SHAP importance — numerical reproduction")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "RELAX_global_SHAP_importance.png", dpi=240)
plt.show()

metrics = ["spearman_rank_correlation", "kendall_rank_correlation", "top_5_jaccard"]
plt.figure(figsize=(8, 5))
plt.boxplot([relax_pairs[m].dropna().to_numpy() for m in metrics],
            tick_labels=["Spearman", "Kendall", "Top-5 Jaccard"])
plt.ylim(-1.05, 1.05)
plt.ylabel("Pairwise fold agreement")
plt.title("RELAX: global explanation stability — numerical reproduction")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "RELAX_global_SHAP_stability_metrics.png", dpi=240)
plt.show()

## 7. Compare the new run against immutable archived references

The comparison checks fold membership counts, fold-specific importance values, pairwise metrics, and overall summary values. Numerical equality is evaluated separately from structural integrity. Small floating-point differences may occur across software builds; the tables report the actual differences rather than hiding them.

In [ ]:
ref_fold = pd.read_csv(REFERENCE_DIR / "RELAX_global_SHAP_fold_audit.csv")
ref_fi = pd.read_csv(REFERENCE_DIR / "RELAX_global_SHAP_importance_by_fold.csv")
ref_pairs = pd.read_csv(REFERENCE_DIR / "RELAX_global_SHAP_pairwise_stability.csv")
ref_summary = pd.read_csv(REFERENCE_DIR / "RELAX_global_SHAP_summary.csv")

# Structural fold comparison
fold_compare = ref_fold.merge(relax_fold_audit, on=["dataset", "fold"], suffixes=("_archived", "_rerun"), validate="one_to_one")
fold_count_cols = ["train_rows", "test_rows", "train_participants", "test_participants", "n_estimators", "seed"]
for col in fold_count_cols:
    fold_compare[f"{col}_match"] = fold_compare[f"{col}_archived"] == fold_compare[f"{col}_rerun"]
fold_compare.to_csv(OUTPUT_DIR / "RELAX_archived_vs_rerun_fold_comparison.csv", index=False)

# Feature importance comparison at each fold-feature pair
fi_compare = ref_fi.merge(relax_fi, on=["fold", "feature"], suffixes=("_archived", "_rerun"), validate="one_to_one")
fi_compare["absolute_difference"] = (fi_compare["mean_abs_shap_archived"] - fi_compare["mean_abs_shap_rerun"]).abs()
fi_compare["relative_difference"] = fi_compare["absolute_difference"] / fi_compare["mean_abs_shap_archived"].abs().replace(0, np.nan)
fi_compare["rank_match"] = fi_compare["rank_archived"] == fi_compare["rank_rerun"]
fi_compare.to_csv(OUTPUT_DIR / "RELAX_archived_vs_rerun_importance_comparison.csv", index=False)

# Pairwise metrics comparison
pair_compare = ref_pairs.merge(relax_pairs, on=["dataset", "fold_a", "fold_b"], suffixes=("_archived", "_rerun"), validate="one_to_one")
for metric in ["spearman_rank_correlation", "kendall_rank_correlation", "top_5_jaccard"]:
    pair_compare[f"{metric}_absolute_difference"] = (pair_compare[f"{metric}_archived"] - pair_compare[f"{metric}_rerun"]).abs()
pair_compare.to_csv(OUTPUT_DIR / "RELAX_archived_vs_rerun_pairwise_comparison.csv", index=False)

summary_compare = ref_summary.merge(relax_summary, on="feature", suffixes=("_archived", "_rerun"), validate="one_to_one")
summary_compare["mean_abs_shap_absolute_difference"] = (summary_compare["mean_abs_shap_archived"] - summary_compare["mean_abs_shap_rerun"]).abs()
summary_compare.to_csv(OUTPUT_DIR / "RELAX_archived_vs_rerun_summary_comparison.csv", index=False)

print("Fold structure comparison:"); display(fold_compare)
print("Feature importance comparison diagnostics:")
display(fi_compare[["absolute_difference", "relative_difference", "rank_match"]].describe())
print("Pairwise metrics comparison:"); display(pair_compare)
print("Archived versus rerun aggregate metrics:")
for label, frame in [("ARCHIVED", ref_pairs), ("RERUN", relax_pairs)]:
    print(label, {m: float(frame[m].mean()) for m in ["spearman_rank_correlation", "kendall_rank_correlation", "top_5_jaccard"]})

## 8. Numerical consistency checks and interpretation

A successful run does not automatically establish exact reproduction. The environment match is evaluated against the recorded components. NumPy and SciPy are explicitly marked as historically undocumented.

In [ ]:
checks = []
def add_check(name, passed, detail):
    checks.append({"check": name, "passed": bool(passed), "detail": str(detail)})

# Fold partition and participant disjointness
test_sets = []
for fold, (tr, te) in enumerate(relax_splits, start=1):
    test_sets.append(set(g_relax[te]))
    add_check(f"Fold {fold}: no train/test participant overlap",
              set(g_relax[tr]).isdisjoint(set(g_relax[te])),
              f"train participants={len(set(g_relax[tr]))}; test participants={len(set(g_relax[te]))}")
add_check("No participant appears in more than one held-out fold",
          all(test_sets[i].isdisjoint(test_sets[j]) for i,j in itertools.combinations(range(5),2)),
          "GroupKFold held-out participant sets are mutually disjoint")
add_check("All eligible rows explained exactly once",
          len(relax_global_shap_rows) == len(relax) and pd.DataFrame(relax_global_shap_rows)["row_index"].nunique() == len(relax),
          f"SHAP rows={len(relax_global_shap_rows)}; eligible rows={len(relax)}")
shap_cols = [c for c in pd.DataFrame(relax_global_shap_rows).columns if c.startswith("shap__")]
add_check("All held-out SHAP values finite",
          np.isfinite(pd.DataFrame(relax_global_shap_rows)[shap_cols].to_numpy()).all(),
          f"SHAP matrix shape={(len(relax_global_shap_rows), len(shap_cols))}")
add_check("Expected 400-tree RF in all folds",
          (relax_fold_audit["n_estimators"] == 400).all(), "400 trees per fold")
add_check("Fold sizes match archived reference",
          all(fold_compare[f"{c}_match"].all() for c in ["train_rows", "test_rows", "train_participants", "test_participants"]),
          "Train/test rows and participant counts")
add_check("Fold seeds match archived reference",
          fold_compare["seed_match"].all(), "Expected seeds 20261014–20261018")
add_check("Input hash matches frozen reference", input_sha256 == EXPECTED_SHA256, input_sha256)

version_matches = {
    "python": CURRENT_ENV["python"] == ORIGINAL_ENV["python"],
    "pandas": CURRENT_ENV["pandas"] == ORIGINAL_ENV["pandas"],
    "scikit_learn": CURRENT_ENV["scikit_learn"] == ORIGINAL_ENV["scikit_learn"],
    "shap": CURRENT_ENV["shap"] == ORIGINAL_ENV["shap"]
}
for k,v in version_matches.items():
    add_check(f"Recorded {k} version match", v, f"current={CURRENT_ENV[k]}; recorded={ORIGINAL_ENV[k]}")

checks_df = pd.DataFrame(checks)
checks_df.to_csv(OUTPUT_DIR / "RELAX_global_SHAP_integrity_checks.csv", index=False)
checks_df.to_csv(OUTPUT_DIR / "RELAX_global_SHAP_integrity_checks.csv", index=False)
display(checks_df)

all_structural = checks_df.loc[~checks_df["check"].str.contains("version match"), "passed"].all()
all_recorded_versions = all(version_matches.values())
if all_structural and all_recorded_versions:
    reproduction_status = "RECORDED-VERSION REPRODUCTION: structural checks and all recorded software versions match. Compare numerical outputs before claiming numerical reproduction."
elif all_structural:
    reproduction_status = "STRUCTURALLY VALID DIAGNOSTIC RERUN: data, folds and numerical consistency checks pass, but one or more recorded software versions differ. Do not call this exact historical reproduction."
else:
    reproduction_status = "FAILED OR INCOMPLETE RUN: one or more structural checks failed. Do not use these outputs in the manuscript."
print("\nRUN STATUS:", reproduction_status)

## 9. Save metadata and package all outputs

The output package includes the newly generated results, environment record, numerical consistency checks, and archived-reference comparisons. It does not replace the original Phase III archive.

In [ ]:
metadata = {
    "analysis": "RELAX global SHAP numerical reproduction",
    "source_method": "RELAX global section of PHASE_III_Model_Aligned_SHAP_Stability_FIXED.ipynb",
    "input_file": DATA_PATH.name, "input_sha256": input_sha256,
    "input_shape": list(relax.shape), "participants": int(pd.Series(g_relax).nunique()),
    "predictors_before_missing_indicators": len(relax_features),
    "rf": RELAX_RF, "n_splits": N_SPLITS, "top_k": TOP_K,
    "seed_base": SEED_RELAX_GLOBAL, "fold_seeds": [SEED_RELAX_GLOBAL+i for i in range(1, N_SPLITS+1)],
    "environment": CURRENT_ENV, "recorded_original_environment": ORIGINAL_ENV,
    "version_matches": version_matches, "reproduction_status": reproduction_status,
    "integrity_checks_passed": int(checks_df["passed"].sum()),
    "integrity_checks_total": int(len(checks_df)),
    "archived_reference_files_are_immutable": True
}
(OUTPUT_DIR / "RELAX_global_SHAP_run_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
(OUTPUT_DIR / "RUN_STATUS.txt").write_text(reproduction_status + "\n", encoding="utf-8")

# Compact human-readable comparison summary
summary_lines = [
    "RELAX GLOBAL SHAP — NUMERICAL REPRODUCTION",
    "="*55,
    reproduction_status,
    "", "ENVIRONMENT", json.dumps(CURRENT_ENV, indent=2),
    "", "ARCHIVED MEAN PAIRWISE METRICS",
    json.dumps({m: float(ref_pairs[m].mean()) for m in ["spearman_rank_correlation", "kendall_rank_correlation", "top_5_jaccard"]}, indent=2),
    "", "RERUN MEAN PAIRWISE METRICS",
    json.dumps({m: float(relax_pairs[m].mean()) for m in ["spearman_rank_correlation", "kendall_rank_correlation", "top_5_jaccard"]}, indent=2),
    "", "NUMERICAL CONSISTENCY CHECKS", checks_df.to_string(index=False)
]
(OUTPUT_DIR / "RELAX_global_SHAP_run_report.txt").write_text("\n".join(summary_lines), encoding="utf-8")

RESULT_ZIP = WORK_DIR / "RELAX_Global_SHAP_Notebook_Results.zip"
with zipfile.ZipFile(RESULT_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUTPUT_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=p.relative_to(OUTPUT_DIR))
print("Created results package:", RESULT_ZIP, "| bytes:", RESULT_ZIP.stat().st_size)

## 10. Download the results package

Upload the generated ZIP back into the research conversation for independent review. Include the full ZIP even if the run status is diagnostic or failed; the metadata and logs are needed to identify the cause.

In [ ]:
try:
    from google.colab import files
    files.download(str(RESULT_ZIP))
except ImportError:
    print("Results saved to:", RESULT_ZIP.resolve())